# Кейс 4.2 · Демо Change Detection (Gradio)

Інтерфейс для фінальної демонстрації: завантажуєте **два вирівняні знімки «До» / «Після»** → модель BIT-CD (навчена в основному ноутбуці `EPS_ChangeDetection_BITCD_Semantic_UA.ipynb`) знаходить зміни та їх тип:
**зруйнований об'єкт · нова вирва · свіжа фортифікація · нова забудова (інша зміна)**.

**На виході:** знімок «Після» з рамками (номер, клас, впевненість), кольорова маска класів, карта ймовірності зміни, таблиця об'єктів (площа в px і м², рамка, впевненість), час обробки та файли для завантаження (JSON + PNG).

**Що потрібно:** `Runtime → T4 GPU` та доступ до Інтернету. За замовчуванням `WEIGHTS_SOURCE='shared'`: ноутбук сам завантажує перевірені `best.pt` і `threshold.json` команди з публічних посилань Google Drive. Особистий Drive, навчальні дані й повторне навчання не потрібні. Час першого запуску залежить від завантаження залежностей і ваг.

**Власні ваги / приклади:** встановіть `WEIGHTS_SOURCE='drive'` у першій комірці. У Colab підключиться Drive; файли очікуються в `MyDrive/EPS_2026/runs/bitcd_semantic_v7/`, приклади — у `MyDrive/EPS_2026/demo_pairs/`.

**Як запускати:** `Runtime → Run all`. Унизу з'явиться інтерфейс (і публічне посилання `*.gradio.live` — зручно відкрити на проєкторі).

> Режим «🧪 Синтетичні вирви та траншеї» вставляє згенеровані об'єкти у ваш знімок «Після» і показує їхню розмітку поруч із прогнозом. Це **прозора демонстрація** класів 2–3, для яких немає реальних розмічених знімків — на демо варто прямо так і пояснювати.

In [ ]:
# ==============================================================================
# 1. Залежності, Google Drive, налаштування
# ==============================================================================
import os, sys, json, time, math, shutil, zipfile, re, glob, subprocess, warnings
from pathlib import Path
from collections import Counter

try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'gradio>=4.40,<6'], check=True)
    import gradio as gr

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

warnings.filterwarnings('ignore', category=UserWarning)
cv2.setNumThreads(0)

# Публічні файли команди; SHA-256 фіксує перевірену версію кожного артефакту.
SHARED_FILES = {
    'best.pt': ('1jtceZDX4cNY9udOLS_6zsN8ptSNVQozj',
                'aabcc6c299fe5e2b75aa452ba9aa4397c639f3aeedc9bffc4351c97f589d938a'),
    'threshold.json': ('1RAjId11DcGvu1hq0K4JN4hJhDT32R1Bs',
                       'c98e9f2610c6afb6c7653222520809a16881c87249b0a605e46bdd0e003dfe64'),
    'BIT_LEVIR_best_ckpt.pt': ('1Qoh-0_Ub3xOdwRphfT_I4lXksm9nEZMK',
                             'c159ba76143447f58c9f367ce8126a0014f2e4ba218cdb97cca173952c38cb3b'),
}


def _download_shared_file(target, file_id, expected_sha256):
    """Кешований публічний файл Drive: SHA-256 перевіряється до заміни цільового файлу."""
    import hashlib

    target = Path(target)

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    if target.is_file() and digest(target) == expected_sha256:
        print('Перевірений кеш:', target.name)
        return target

    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'gdown>=5.2,<6'], check=True)
        import gdown

    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + '.download')
    try:
        print('Завантаження:', target.name)
        result = gdown.download(id=file_id, output=str(temporary), quiet=False, use_cookies=False)
        if result is None or not temporary.is_file():
            raise RuntimeError('Google Drive не повернув файл.')
        if digest(temporary) != expected_sha256:
            raise ValueError('SHA-256 не збігається з перевіреною версією файлу.')
        os.replace(temporary, target)
    except Exception as exc:
        raise RuntimeError(
            f'Не вдалося отримати {target.name}: {exc} '
            'Перевірте доступ за посиланням, ліміт завантажень Google Drive '
            'або використайте власні файли за інструкцією README.'
        ) from exc
    finally:
        temporary.unlink(missing_ok=True)
    return target


# 'shared': автоматичне завантаження публічних ваг, без підключення особистого Drive.
# 'drive': власні best.pt / threshold.json та demo_pairs у MyDrive/EPS_2026.
WEIGHTS_SOURCE = 'shared'
if WEIGHTS_SOURCE not in ('shared', 'drive'):
    raise ValueError("WEIGHTS_SOURCE має бути 'shared' або 'drive'.")

IN_COLAB = 'google.colab' in sys.modules
WORK = Path('/content/eps_demo') if IN_COLAB else Path(os.environ.get('EPS_WORK', './eps_demo_work'))
if WEIGHTS_SOURCE == 'drive':
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_ROOT = Path('/content/drive/MyDrive/EPS_2026')
    else:
        DRIVE_ROOT = Path(os.environ.get('EPS_DRIVE_ROOT', './EPS_2026'))
else:
    DRIVE_ROOT = WORK / 'assets'
RUN_DIR = DRIVE_ROOT / 'runs' / 'bitcd_semantic_v7'      # куди основний ноутбук зберіг модель
BEST_PT = RUN_DIR / 'best.pt'
THR_JSON = RUN_DIR / 'threshold.json'
DEMO_PAIRS = DRIVE_ROOT / 'demo_pairs'                    # (необов'язково) приклади, експортовані з основного ноутбука
FOT_INSTANCES_ZIP = DRIVE_ROOT / 'FOTBCD-Instances.zip'   # запасне джерело прикладів
OUT_DIR = WORK / 'outputs'
for d in (WORK, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

if WEIGHTS_SOURCE == 'shared':
    _download_shared_file(BEST_PT, *SHARED_FILES['best.pt'])
    _download_shared_file(THR_JSON, *SHARED_FILES['threshold.json'])

# ---- ті самі константи, що й в основному ноутбуці
LBL_BG, LBL_DESTROYED, LBL_CRATER, LBL_FORT, LBL_NEWBLD = 0, 1, 2, 3, 4
LBL_UNKNOWN, LBL_IGNORE = 254, 255
CLASS_NAMES = {1: "зруйнований об'єкт", 2: 'нова вирва', 3: 'свіжа фортифікація', 4: 'нова забудова (інша зміна)'}
CLASS_SHORT = {1: 'destroyed', 2: 'crater', 3: 'fortification', 4: 'new_building'}
CLASS_UA_SHORT = {1: 'зруйн.', 2: 'вирва', 3: 'фортиф.', 4: 'нова забуд.'}
CLASS_COLORS = {1: (230, 25, 75), 2: (255, 140, 0), 3: (255, 225, 25), 4: (0, 130, 200)}
N_TYPES = 4
TARGET_CLASSES = [1, 2, 3]
TILE, STRIDE, INFER_BATCH, AMP, TTA = 256, 192, 32, True, False
MIN_OBJ_AREA_M2 = {1: 6.0, 2: 0.4, 3: 0.6, 4: 6.0}
DEFAULT_GSD_UNKNOWN = 0.3          # для фільтра мінімальної площі, якщо GSD знімка невідомий
FONT_PATH = os.path.join(matplotlib.get_data_path(), 'fonts', 'ttf', 'DejaVuSans.ttf')   # кирилиця на рамках

print('Gradio', gr.__version__, '| Colab:', IN_COLAB, '| модель:', BEST_PT, '| існує:', BEST_PT.exists())

In [ ]:
# ==============================================================================
# 2. Модель BIT-CD (2 голови) — той самий код, що в основному ноутбуці; ваги з best.pt
# ==============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
DEVICE_NAME = torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'
print('PyTorch', torch.__version__, '| пристрій:', DEVICE_NAME)
if DEVICE.type != 'cuda':
    print('⚠ GPU не підключено (Runtime → T4 GPU). На CPU працює, але кадр 1024×1024 обробляється ~5–15 с.')


# ---- блоки трансформера (імена параметрів = офіційний BIT_CD, щоб ваги завантажились строго)
class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__(); self.fn = fn

    def forward(self, x, **kw):
        return self.fn(x, **kw) + x


class Residual2(nn.Module):
    def __init__(self, fn):
        super().__init__(); self.fn = fn

    def forward(self, x, x2, **kw):
        return self.fn(x, x2, **kw) + x


class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__(); self.norm = nn.LayerNorm(dim); self.fn = fn

    def forward(self, x, **kw):
        return self.fn(self.norm(x), **kw)


class PreNorm2(nn.Module):
    def __init__(self, dim, fn):
        super().__init__(); self.norm = nn.LayerNorm(dim); self.fn = fn

    def forward(self, x, x2, **kw):
        return self.fn(self.norm(x), self.norm(x2), **kw)


class FeedForward(nn.Module):
    def __init__(self, dim, hidden, dropout=0.):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, dim), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


def _split_heads(t, h):            # 'b n (h d) -> b h n d'
    b, n, _ = t.shape
    return t.reshape(b, n, h, -1).transpose(1, 2)


def _merge_heads(t):               # 'b h n d -> b n (h d)'
    b, h, n, d = t.shape
    return t.transpose(1, 2).reshape(b, n, h * d)


class Attention(nn.Module):
    def __init__(self, dim, heads=8, dim_head=64, dropout=0.):
        super().__init__()
        inner = dim_head * heads
        self.heads, self.scale = heads, dim ** -0.5          # як в оригіналі BIT: масштаб від dim
        self.to_qkv = nn.Linear(dim, inner * 3, bias=False)
        self.to_out = nn.Sequential(nn.Linear(inner, dim), nn.Dropout(dropout))

    def forward(self, x, mask=None):
        q, k, v = (_split_heads(t, self.heads) for t in self.to_qkv(x).chunk(3, dim=-1))
        attn = (torch.einsum('bhid,bhjd->bhij', q, k) * self.scale).softmax(dim=-1)
        return self.to_out(_merge_heads(torch.einsum('bhij,bhjd->bhid', attn, v)))


class CrossAttention(nn.Module):
    def __init__(self, dim, heads=8, dim_head=64, dropout=0., softmax=True):
        super().__init__()
        inner = dim_head * heads
        self.heads, self.scale, self.softmax = heads, dim ** -0.5, softmax
        self.to_q = nn.Linear(dim, inner, bias=False)
        self.to_k = nn.Linear(dim, inner, bias=False)
        self.to_v = nn.Linear(dim, inner, bias=False)
        self.to_out = nn.Sequential(nn.Linear(inner, dim), nn.Dropout(dropout))

    def forward(self, x, m, mask=None):
        q, k, v = _split_heads(self.to_q(x), self.heads), _split_heads(self.to_k(m), self.heads), _split_heads(self.to_v(m), self.heads)
        dots = torch.einsum('bhid,bhjd->bhij', q, k) * self.scale
        attn = dots.softmax(dim=-1) if self.softmax else dots
        return self.to_out(_merge_heads(torch.einsum('bhij,bhjd->bhid', attn, v)))


class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout=0.):
        super().__init__()
        self.layers = nn.ModuleList([nn.ModuleList([
            Residual(PreNorm(dim, Attention(dim, heads, dim_head, dropout))),
            Residual(PreNorm(dim, FeedForward(dim, mlp_dim, dropout)))]) for _ in range(depth)])

    def forward(self, x):
        for attn, ff in self.layers:
            x = ff(attn(x))
        return x


class TransformerDecoder(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout=0., softmax=True):
        super().__init__()
        self.layers = nn.ModuleList([nn.ModuleList([
            Residual2(PreNorm2(dim, CrossAttention(dim, heads, dim_head, dropout, softmax))),
            Residual(PreNorm(dim, FeedForward(dim, mlp_dim, dropout)))]) for _ in range(depth)])

    def forward(self, x, m):
        for attn, ff in self.layers:
            x = ff(attn(x, m))
        return x


def two_layer_conv(cin, cout, k=3):
    return nn.Sequential(nn.Conv2d(cin, cin, k, padding=k // 2, bias=False), nn.BatchNorm2d(cin), nn.ReLU(),
                         nn.Conv2d(cin, cout, k, padding=k // 2))


class BITSemantic(nn.Module):
    """BIT-CD base_transformer_pos_s4_dd8_dedim8 + голова типу зміни."""

    def __init__(self, n_types=N_TYPES, token_len=4, dec_depth=8, dec_dim_head=8):
        super().__init__()
        self.resnet = torchvision.models.resnet18(weights=None)
        # офіційний BIT: replace_stride_with_dilation=[False, True, True]; у BasicBlock дилатація ігнорується,
        # тож еквівалент — прибрати stride у layer3/layer4
        for layer in (self.resnet.layer3, self.resnet.layer4):
            layer[0].conv1.stride = (1, 1)
            layer[0].downsample[0].stride = (1, 1)
        self.upsamplex2 = nn.Upsample(scale_factor=2)
        self.upsamplex4 = nn.Upsample(scale_factor=4, mode='bilinear')
        self.conv_pred = nn.Conv2d(256, 32, kernel_size=3, padding=1)
        self.classifier = two_layer_conv(32, 2)                    # голова «є/немає зміни» (LEVIR)
        self.token_len = token_len
        self.conv_a = nn.Conv2d(32, token_len, kernel_size=1, padding=0, bias=False)
        self.pos_embedding = nn.Parameter(torch.randn(1, token_len * 2, 32))
        self.transformer = Transformer(dim=32, depth=1, heads=8, dim_head=64, mlp_dim=64)
        self.transformer_decoder = TransformerDecoder(dim=32, depth=dec_depth, heads=8, dim_head=dec_dim_head, mlp_dim=64)
        self.type_head = nn.Sequential(                            # нова спрямована голова типу
            nn.Conv2d(96, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, n_types, 1))

    def forward_single(self, x):
        r = self.resnet
        x = r.maxpool(r.relu(r.bn1(r.conv1(x))))
        x = r.layer3(r.layer2(r.layer1(x)))                      # 1/8, 256 каналів
        return self.conv_pred(self.upsamplex2(x))                  # 1/4, 32 канали

    def tokens(self, x):
        b, c, h, w = x.shape
        att = torch.softmax(self.conv_a(x).view(b, self.token_len, -1), dim=-1)
        return torch.einsum('bln,bcn->blc', att, x.view(b, c, -1))

    def decode(self, x, m):
        b, c, h, w = x.shape
        y = self.transformer_decoder(x.flatten(2).transpose(1, 2), m)
        return y.transpose(1, 2).reshape(b, c, h, w)

    def forward(self, x1, x2):
        f1, f2 = self.forward_single(x1), self.forward_single(x2)
        t = self.transformer(torch.cat([self.tokens(f1), self.tokens(f2)], dim=1) + self.pos_embedding)
        t1, t2 = t.chunk(2, dim=1)
        f1, f2 = self.decode(f1, t1), self.decode(f2, t2)
        change = self.classifier(self.upsamplex4(torch.abs(f1 - f2)))
        typ = self.type_head(torch.cat([f1, f2, f2 - f1], dim=1))
        typ = F.interpolate(typ, size=change.shape[-2:], mode='bilinear', align_corners=False)
        return change, typ


def load_demo_model():
    if not BEST_PT.exists():
        raise FileNotFoundError(f'Не знайдено {BEST_PT}. Перевірте RUN_DIR або скопіюйте best.pt з основного ноутбука на Drive.')
    ck = torch.load(BEST_PT, map_location='cpu', weights_only=False)     # власний checkpoint основного ноутбука
    m = BITSemantic()
    m.load_state_dict(ck['model'], strict=True)
    m.to(DEVICE).eval()
    thr = 0.5
    if THR_JSON.exists():
        threshold_config = json.loads(THR_JSON.read_text())
        thr = float(threshold_config['threshold'])
        if not 0 <= thr <= 1:
            raise ValueError('threshold.json: threshold має бути в межах [0, 1].')
        if threshold_config.get('epoch') is not None and ck.get('epoch') is not None:
            if threshold_config['epoch'] != ck['epoch']:
                raise ValueError('best.pt і threshold.json належать до різних епох. Використайте узгоджену пару файлів.')
    else:
        print('⚠ threshold.json не знайдено — поріг 0.5 (у звіті основного ноутбука поріг обирався на validation).')
    return m, ck, thr


MODEL, CKPT, THRESHOLD = load_demo_model()
print(f"Модель: epoch {CKPT.get('epoch')} | val score {CKPT.get('score', float('nan')):.4f} | поріг {THRESHOLD:.2f}")

In [ ]:
# ==============================================================================
# 3. Інференс кадру довільного розміру + постобробка (той самий код, що в основному ноутбуці)
# ==============================================================================
_WIN_CACHE = {}


def _blend_window(tile):
    if tile not in _WIN_CACHE:
        h = 0.1 + 0.9 * torch.sin(math.pi * (torch.arange(tile, dtype=torch.float32) + 0.5) / tile)
        _WIN_CACHE[tile] = (h[:, None] * h[None, :]).to(DEVICE)
    return _WIN_CACHE[tile]


def _forward_probs(model, a, b, tta):
    """Ймовірності зміни (B×H×W) і типів (B×T×H×W) з опційним TTA (віддзеркалення)."""
    flips = [[]] + ([[3], [2], [2, 3]] if tta else [])
    pc_sum, pt_sum = 0, 0
    for f in flips:
        aa, bb = (a.flip(f), b.flip(f)) if f else (a, b)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(AMP and DEVICE.type == 'cuda')):
            ch, ty = model(aa, bb)
        pc, pt = torch.softmax(ch.float(), 1)[:, 1], torch.softmax(ty.float(), 1)
        if f:
            pc, pt = pc.flip([d - 1 for d in f]), pt.flip(f)
        pc_sum, pt_sum = pc_sum + pc, pt_sum + pt
    return pc_sum / len(flips), pt_sum / len(flips)


@torch.inference_mode()
def predict_full(model, pre, post, stride=STRIDE, tta=TTA, batch=INFER_BATCH, return_type_prob=False):
    """pre/post: H×W×3 uint8 (numpy) → prob [H,W] float32, typ [H,W] uint8 (індекс 0..3 ⇒ клас 1..4)."""
    model.eval()
    H, W = pre.shape[:2]
    ph, pw = max(0, TILE - H), max(0, TILE - W)
    if ph or pw:
        pre = cv2.copyMakeBorder(pre, 0, ph, 0, pw, cv2.BORDER_REFLECT_101)
        post = cv2.copyMakeBorder(post, 0, ph, 0, pw, cv2.BORDER_REFLECT_101)
    Hp, Wp = pre.shape[:2]
    a = torch.from_numpy(np.ascontiguousarray(pre)).to(DEVICE).permute(2, 0, 1).float().div_(255).sub_(0.5).div_(0.5)
    b = torch.from_numpy(np.ascontiguousarray(post)).to(DEVICE).permute(2, 0, 1).float().div_(255).sub_(0.5).div_(0.5)
    coords = [(y, x) for y in window_origins(Hp, TILE, stride) for x in window_origins(Wp, TILE, stride)]
    win = _blend_window(TILE)
    prob = torch.zeros(Hp, Wp, device=DEVICE)
    tsum = torch.zeros(N_TYPES, Hp, Wp, device=DEVICE)
    wsum = torch.zeros(Hp, Wp, device=DEVICE)
    for i in range(0, len(coords), batch):
        cs = coords[i:i + batch]
        pa = torch.stack([a[:, y:y + TILE, x:x + TILE] for y, x in cs])
        pb = torch.stack([b[:, y:y + TILE, x:x + TILE] for y, x in cs])
        pc, pt = _forward_probs(model, pa, pb, tta)
        for j, (y, x) in enumerate(cs):
            prob[y:y + TILE, x:x + TILE] += pc[j] * win
            tsum[:, y:y + TILE, x:x + TILE] += pt[j] * win
            wsum[y:y + TILE, x:x + TILE] += win
    prob = (prob / wsum)[:H, :W]
    tprob = (tsum / wsum)[:, :H, :W]
    typ = tprob.argmax(0).to(torch.uint8)
    out = (prob.float().cpu().numpy(), typ.cpu().numpy())
    if return_type_prob:
        out = out + (tprob.max(0).values.float().cpu().numpy(),)
    return out


def cuda_sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


def window_origins(length, win, stride):
    if length <= win:
        return [0]
    pos = list(range(0, length - win + 1, stride))
    if pos[-1] + win < length:
        pos.append(length - win)
    return pos


def pred_class_map(prob, typ, thr):
    return np.where(prob >= thr, typ.astype(np.uint8) + 1, 0).astype(np.uint8)


def extract_objects(cls_map, prob, conf, gsd=None):
    g = gsd or DEFAULT_GSD_UNKNOWN
    objs = []
    for c in (1, 2, 3, 4):
        m = cls_map == c
        if not m.any():
            continue
        n, comp, stats, cent = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
        for k in range(1, n):
            x, y, w, h, area = (int(v) for v in stats[k])
            if area < MIN_OBJ_AREA_M2[c] / g ** 2:
                continue
            mk = comp[y:y + h, x:x + w] == k
            score = float(prob[y:y + h, x:x + w][mk].mean() * conf[y:y + h, x:x + w][mk].mean())
            objs.append(dict(cls=c, name=CLASS_NAMES[c], target=c in TARGET_CLASSES, area_px=area,
                             area_m2=round(area * gsd ** 2, 2) if gsd else None, bbox_xywh=[x, y, w, h],
                             centroid_xy=[round(float(cent[k][0]), 1), round(float(cent[k][1]), 1)], score=round(score, 3)))
    objs = sorted(objs, key=lambda o: -o['score'])
    for i, o in enumerate(objs, 1):
        o['id'] = i
    return objs


def label_overlay(rgb, label, alpha=0.5):
    out = rgb.astype(np.float32).copy()
    for c, col in CLASS_COLORS.items():
        m = label == c
        out[m] = out[m] * (1 - alpha) + np.array(col, np.float32) * alpha
    return out.clip(0, 255).astype(np.uint8)


def read_rgb(path):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)          # PNG/JPG/TIFF (16-біт → 8-біт)
    if bgr is None:
        raise ValueError(f'Не вдалося прочитати зображення: {Path(str(path)).name}')
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def read_gray(path):
    m = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if m is None:
        raise ValueError(f'Не вдалося прочитати маску: {path}')
    return m[..., 0] if m.ndim == 3 else m


# прогрів cudnn на кадрі 1024×1024
_z = np.zeros((1024, 1024, 3), np.uint8)
predict_full(MODEL, _z, _z); cuda_sync()
print('Інференс готовий.')

In [ ]:
# ==============================================================================
# 4. Синтетичний генератор (той самий, що в основному ноутбуці) — для режиму «🧪»
# ==============================================================================
# Принципи (відповідь на критику v3):
#   * без мозаїк: беремо ЦІЛІ реальні пари «до/після» (різні дати, реальні сезонні відмінності);
#   * об'єкти рендеряться через карту висот: чаша + вал + рваний викид ґрунту, освітлення
#     від сонця (hillshade) і ВІДКИНУТІ тіні; колір ґрунту адаптується до яскравості сцени;
#     шум і різкість підганяються під знімок;
#   * розміри задаються в МЕТРАХ і переводяться в пікселі через GSD знімка;
#   * «старі» об'єкти малюються на ОБОХ знімках з міткою 0 (без змін) — контроль того,
#     що модель вчить саме ЗМІНУ, а не артефакти генератора;
#   * окремий режим "shift" (відкладені палітри/параметри) для чесного тесту переносу.
# (numpy, cv2 та константи міток LBL_* уже визначені в конфігурації)

# Палітри свіжого ґрунту (RGB, приблизно для середньої яскравості сцени ~115)
SOIL_PALETTES = {
    # 'A' — для train / val / test-ID
    'A': {'dark_brown': (88, 72, 58), 'brown': (122, 100, 80), 'grey_brown': (118, 108, 98),
          'tan': (158, 136, 108), 'ochre': (140, 118, 82)},
    # 'B' — ВІДКЛАДЕНА палітра лише для test-shift (у навчанні не зустрічається)
    'B': {'chalk': (196, 190, 176), 'reddish': (148, 98, 74), 'scorched': (62, 58, 55),
          'loess_yellow': (178, 158, 112)},
}

SYN_REGIMES = {
    # діапазони для різних режимів генерації
    'id':    dict(palette='A', sun_elev=(28, 62), crater_diam_mix=((1.2, 2.5, .35), (2.5, 6.0, .50), (6.0, 13.0, .15)),
                  trench_styles=('zigzag', 'crenellated', 'wavy', 'segmented'), p_dragon=0.10, p_ditch=0.12),
    'shift': dict(palette='B', sun_elev=(18, 30), crater_diam_mix=((1.0, 2.0, .25), (2.0, 7.0, .45), (7.0, 16.0, .30)),
                  trench_styles=('zigzag', 'crenellated', 'wavy', 'segmented'), p_dragon=0.18, p_ditch=0.18),
}


# ------------------------------------------------------------------ шуми / утиліти
def _smoothstep(e0, e1, x):
    t = np.clip((x - e0) / (e1 - e0 + 1e-9), 0, 1)
    return t * t * (3 - 2 * t)


def fractal_noise(h, w, rng, sigmas=(1.0, 2.5, 6.0, 14.0), weights=(0.15, 0.3, 0.35, 0.2)):
    """Гладкий багатомасштабний шум, нормований до std≈1."""
    out = np.zeros((h, w), np.float32)
    for s, wt in zip(sigmas, weights):
        n = rng.standard_normal((h, w)).astype(np.float32)
        n = cv2.GaussianBlur(n, (0, 0), s)
        out += wt * n / (n.std() + 1e-6)
    return out / (out.std() + 1e-6)


def scene_stats(img):
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    hp = g - cv2.GaussianBlur(g, (0, 0), 1.2)
    noise = 1.4826 * np.median(np.abs(hp - np.median(hp)))
    return {'lum': float(np.median(g)), 'noise': float(np.clip(noise, 0.8, 6.0))}


def ground_maps(img, exclusion=None):
    """Повертає дві карти ймовірностей розміщення:
    natural — природний ґрунт/рослинність (поля, луки, лісосмуги, оголений ґрунт): для траншей і вирв;
    any     — природний ґрунт + слабка вага для сірих покриттів (дороги/майданчики): лише для вирв.
    Дахи (дуже світлі однорідні та сірі/сині поверхні) майже виключаються.
    exclusion — маска, де розміщувати НЕ можна (будівлі, вже розмічені зміни)."""
    f = img.astype(np.float32) / 255.0
    r, g, b = f[..., 0], f[..., 1], f[..., 2]
    exg = (2 * g - r - b) / (r + g + b + 1e-3)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
    hue, sat, val = hsv[..., 0] * 2, hsv[..., 1] / 255, hsv[..., 2] / 255
    veg = _smoothstep(0.0, 0.08, exg)
    soil = ((hue >= 17) & (hue <= 55) & (sat >= 0.10) & (sat <= 0.60) & (val > 0.22) & (val < 0.92)).astype(np.float32)
    natural = np.maximum(veg, soil)
    natural = cv2.GaussianBlur(natural, (0, 0), 3)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.GaussianBlur(np.sqrt(gx * gx + gy * gy), (0, 0), 5)
    calm = np.clip(1.0 - mag / (np.percentile(mag, 97) + 1e-6), 0, 1)
    lum = cv2.GaussianBlur(gray, (0, 0), 3)
    not_extreme = np.clip((lum - 30) / 30, 0, 1) * np.clip((215 - lum) / 35, 0, 1)
    nat = natural * calm ** 1.5 * not_extreme                 # траншеї: відкритий ґрунт/луки, не під кронами
    manmade_flat = (1 - natural) * calm * np.clip((lum - 40) / 30, 0, 1) * np.clip((165 - lum) / 30, 0, 1)
    anyp = natural * (0.25 + 0.75 * calm) * not_extreme + 0.10 * manmade_flat   # вирви: будь-де, переважно на ґрунті
    if exclusion is not None and exclusion.any():
        ex = cv2.dilate(exclusion.astype(np.uint8), np.ones((15, 15), np.uint8)) > 0
        nat[ex] = 0
        anyp[ex] = 0
    return nat.astype(np.float32), anyp.astype(np.float32)


def sample_point(prob, rng, margin=8):
    h, w = prob.shape
    p = prob[margin:h - margin, margin:w - margin].ravel().astype(np.float64)
    s = p.sum()
    if s <= 1e-6:
        return None
    i = rng.choice(p.size, p=p / s)
    y, x = divmod(i, w - 2 * margin)
    return x + margin, y + margin


# ------------------------------------------------------------------ рендер карти висот
class Canvas:
    """Накопичує висоти/покриття ґрунтом для ОДНОГО знімка, потім рендерить за один прохід.
    Так тіні від сусідніх об'єктів (вал вирви, бруствер траншеї) взаємодіють коректно."""

    def __init__(self, img, gsd, sun_az, sun_el, rng, palette_rgb, lum):
        self.img = img
        self.h, self.w = img.shape[:2]
        self.gsd = gsd
        self.rng = rng
        self.height = np.zeros((self.h, self.w), np.float32)    # метри
        self.alpha = np.zeros((self.h, self.w), np.float32)     # покриття свіжим ґрунтом
        self.color = np.zeros((self.h, self.w, 3), np.float32)  # колір ґрунту (передмножений на alpha)
        self.dark = np.zeros((self.h, self.w), np.float32)      # затемнення (вологий ґрунт/кіптява)
        self.sun_az, self.sun_el = np.deg2rad(sun_az), np.deg2rad(sun_el)
        k = np.clip((lum / 115.0) ** 0.7, 0.6, 1.5)
        self.soil = np.array(palette_rgb, np.float32) * k * rng.uniform(0.92, 1.08)
        self.relief = rng.uniform(0.45, 0.8)                  # сила 3D-освітлення (надир: слабка)
        self.shadow_k = rng.uniform(0.40, 0.60)               # глибина тіні (розсіяне світло)

    def add(self, y0, y1, x0, x1, height, alpha, soil_rgb, dark=None):
        sl = (slice(y0, y1), slice(x0, x1))
        self.height[sl] += height
        a_old = self.alpha[sl]
        a_new = np.maximum(a_old, alpha)
        # колір змішуємо пропорційно приросту покриття; soil_rgb — (3,) або карта (h, w, 3)
        gain = np.clip(a_new - a_old, 0, 1)[..., None]
        self.color[sl] = self.color[sl] * (1 - gain) + np.asarray(soil_rgb, np.float32) * gain
        self.alpha[sl] = a_new
        if dark is not None:
            self.dark[sl] = np.maximum(self.dark[sl], dark)

    def render(self, noise_sigma, blur_sigma):
        """Рендер лише в межах активної області (швидко)."""
        act = (np.abs(self.height) > 1e-4) | (self.alpha > 1e-3) | (self.dark > 1e-3)
        if not act.any():
            return self.img.copy()
        ys, xs = np.where(act)
        pad = 90
        y0, y1 = max(0, ys.min() - pad), min(self.h, ys.max() + pad + 1)
        x0, x1 = max(0, xs.min() - pad), min(self.w, xs.max() + pad + 1)
        sub = _SubCanvas(self, y0, y1, x0, x1)
        res = self.img.copy()
        res[y0:y1, x0:x1] = sub._render(noise_sigma, blur_sigma)
        return res

    def _render(self, noise_sigma, blur_sigma):
        img = self.img.astype(np.float32)
        hpx = self.height / self.gsd                      # висота в «пікселях» для нахилів
        hrel = hpx * self.relief                           # нахили для дифузного освітлення (приглушені)
        gy, gx = np.gradient(hrel)
        nx, ny, nz = -gx, -gy, np.ones_like(gx)
        nrm = np.sqrt(nx * nx + ny * ny + 1)
        lx = np.cos(self.sun_el) * np.sin(self.sun_az)
        ly = -np.cos(self.sun_el) * np.cos(self.sun_az)
        lz = np.sin(self.sun_el)
        shade = (nx * lx + ny * ly + nz * lz) / nrm / lz
        shade = np.clip(shade, 0.45, 1.3)
        # відкинуті тіні: «марш» уздовж напрямку на сонце
        shadow = np.zeros_like(hpx, dtype=bool)
        active = np.abs(hpx) > 1e-3
        if active.any():
            dx, dy = np.sin(self.sun_az), -np.cos(self.sun_az)
            tan_el = np.tan(self.sun_el)
            max_steps = int(min(80, np.ceil((hpx.max() - hpx.min()) / max(tan_el, 0.05)) + 1))
            for s in range(1, max_steps + 1):
                ox, oy = int(round(dx * s)), int(round(dy * s))
                shifted = np.full_like(hpx, -1e9)
                ys0, ys1 = max(0, -oy), min(self.h, self.h - oy)
                xs0, xs1 = max(0, -ox), min(self.w, self.w - ox)
                shifted[ys0:ys1, xs0:xs1] = hpx[ys0 + oy:ys1 + oy, xs0 + ox:xs1 + ox]
                shadow |= shifted > hpx + s * tan_el + 0.05
        shadow_f = cv2.GaussianBlur(shadow.astype(np.float32), (0, 0), 0.7)
        # композиція: ґрунт поверх фону (частково зберігаємо текстуру фону)
        a = self.alpha[..., None]
        base_lum = img.mean(2, keepdims=True)
        soil = self.color * (1 + 0.10 * self.tex[..., None])
        # тонка модуляція текстурою фону: ґрунт не «пластиковий»
        soil = soil * (0.85 + 0.15 * base_lum / (base_lum.mean() + 1e-6))
        out = img * (1 - 0.92 * a) + soil * 0.92 * a
        out = out * (1 - 0.45 * self.dark[..., None])
        out = out * shade[..., None]
        out = out * (1 - self.shadow_k * shadow_f[..., None])
        # маска впливу: де змінилась картинка
        infl = np.clip(np.maximum(self.alpha, np.maximum(np.abs(shade - 1) * 3, shadow_f)), 0, 1)
        infl = np.maximum(infl, self.dark)
        infl = cv2.GaussianBlur(infl, (0, 0), 1.0)
        if blur_sigma > 0:
            out = cv2.GaussianBlur(out, (0, 0), blur_sigma)
        noise = self.rng.standard_normal(out.shape).astype(np.float32) * noise_sigma
        out = out + noise * infl[..., None]
        res = img * (1 - infl[..., None]) + out * infl[..., None]
        return np.clip(res, 0, 255).astype(np.uint8)


class _SubCanvas(Canvas):
    """Вирізка полотна (ті самі параметри сонця/ґрунту) для швидкого рендеру."""

    def __init__(self, parent, y0, y1, x0, x1):  # noqa: D401 — без виклику Canvas.__init__
        sl = (slice(y0, y1), slice(x0, x1))
        self.img = parent.img[sl]
        self.h, self.w = self.img.shape[:2]
        self.gsd, self.rng = parent.gsd, parent.rng
        self.height, self.alpha = parent.height[sl], parent.alpha[sl]
        self.color, self.dark = parent.color[sl], parent.dark[sl]
        self.tex = fractal_noise(self.h, self.w, parent.rng, sigmas=(0.7, 1.6, 4.0), weights=(0.4, 0.4, 0.2))
        self.sun_az, self.sun_el = parent.sun_az, parent.sun_el
        self.relief, self.shadow_k, self.soil = parent.relief, parent.shadow_k, parent.soil


# ------------------------------------------------------------------ вирва
def crater_geometry(canvas, cx, cy, diam_m, rng, soil_rgb, old=False):
    """Свіжа вирва у надирному знімку: темна волога/затінена чаша, низький нерівний вал,
    світліший рваний викид ґрунту з «променями» та окремими грудками на відстані до ~4 радіусів."""
    gsd = canvas.gsd
    r = diam_m / 2 / gsd                                     # радіус у пікселях
    ext = rng.uniform(2.2, 3.6)                              # протяжність суцільного викиду (в радіусах)
    R = int(np.ceil(r * (ext + 1.0) / (0.78 * 0.8) + 3))     # з запасом: еліпс і збурення контуру
    x0, x1 = max(0, int(cx) - R), min(canvas.w, int(cx) + R + 1)
    y0, y1 = max(0, int(cy) - R), min(canvas.h, int(cy) + R + 1)
    if x1 - x0 < 3 or y1 - y0 < 3:
        return None
    yy, xx = np.mgrid[y0:y1, x0:x1].astype(np.float32)
    dx, dy = xx - cx, yy - cy
    th, ecc = rng.uniform(0, np.pi), rng.uniform(0.78, 1.0)
    u = dx * np.cos(th) + dy * np.sin(th)
    v = (-dx * np.sin(th) + dy * np.cos(th)) / ecc
    rho = np.sqrt(u * u + v * v) / max(r, 0.5)
    phi = np.arctan2(v, u)
    hh, ww = rho.shape
    pert = np.zeros_like(rho)
    for k in range(2, 9):                                    # нерівний контур чаші
        pert += rng.normal(0, 0.06 / k ** 0.5) * np.cos(k * phi + rng.uniform(0, 2 * np.pi))
    pert = np.clip(pert, -0.2, 0.2) * min(1.0, r / 10.0)     # малі вирви — майже круглі
    edge_noise = fractal_noise(hh, ww, rng, sigmas=(max(1.2, r / 6), max(1.5, r / 3)), weights=(0.5, 0.5))
    rho_p = rho * (1 + pert) * (1 + 0.04 * edge_noise)
    # висоти в метрах: неглибока чаша (d/D≈0.12–0.22), низький нерівний вал
    depth = diam_m * rng.uniform(0.12, 0.22)
    rim_h = depth * rng.uniform(0.12, 0.30) * (1 + 0.2 * edge_noise)
    bowl = -depth * np.clip(1 - rho_p ** 2, 0, 1) ** 0.9
    rim = rim_h * np.exp(-((rho_p - 1.0) / 0.25) ** 2)
    height = bowl + rim
    # викид: промені + великі плями, спадає з відстанню
    n_rays = int(rng.integers(6, 16))
    rays = np.zeros_like(phi)
    for _ in range(n_rays):
        c, wdt = rng.uniform(-np.pi, np.pi), rng.uniform(0.10, 0.40)
        dphi = (phi - c + np.pi) % (2 * np.pi) - np.pi
        rays = np.maximum(rays, np.exp(-(dphi / wdt) ** 2) * rng.uniform(0.4, 1.0))
    patch = fractal_noise(hh, ww, rng, sigmas=(max(0.8, r / 6), max(1.2, r / 2.5), max(1.5, r)), weights=(0.2, 0.45, 0.35))
    fall = np.clip(1 - (rho_p - 0.9) / (ext - 0.9), 0, 1) ** 1.4
    ej = _smoothstep(0.0, 0.5, fall * (1.2 + 0.8 * rays) + 0.4 * patch - 0.5) * (rho_p > 0.8)
    clods = (fractal_noise(hh, ww, rng, sigmas=(max(0.6, r / 8),), weights=(1.0,)) > 2.5) & (rho_p > 1.1) & (rho_p < ext)
    ej = np.maximum(ej, clods.astype(np.float32) * 0.7 * (rays > 0.25))
    pit = _smoothstep(0.95, 0.70, rho_p)                     # темна яма
    a_bowl = _smoothstep(1.12, 0.85, rho_p)
    alpha = np.maximum(a_bowl, ej * rng.uniform(0.7, 0.95))
    # кольори: світліший викид (підґрунтя) + темна волога/затінена яма
    ej_col = np.asarray(soil_rgb, np.float32) * rng.uniform(1.0, 1.25)
    pit_col = np.asarray(soil_rgb, np.float32) * rng.uniform(0.45, 0.68)
    color = ej_col[None, None, :] * (1 - pit[..., None]) + pit_col[None, None, :] * pit[..., None]
    dark = pit * rng.uniform(0.08, 0.25)
    if rng.random() < 0.3:                                    # кіптява навколо
        dark = np.maximum(dark, _smoothstep(0.2, 0.9, ej) * rng.uniform(0.15, 0.35))
    if old:                                                   # «стара» вирва: заросла, просіла, низький контраст
        height *= rng.uniform(0.3, 0.6)
        alpha = a_bowl * rng.uniform(0.25, 0.5)
        dark *= rng.uniform(0.2, 0.5)
    # мітка: чаша + вал + ЩІЛЬНИЙ близький викид (окремі грудки не розмічаються)
    ej_dense = cv2.GaussianBlur(np.where(clods, 0, ej).astype(np.float32), (0, 0), 1.2)
    label = (rho_p <= 1.15) | ((ej_dense > 0.6) & (rho_p < 1.8))
    return dict(box=(y0, y1, x0, x1), height=height.astype(np.float32), alpha=alpha.astype(np.float32),
                dark=dark.astype(np.float32), label=label, color=color.astype(np.float32))


# ------------------------------------------------------------------ траншеї/фортифікації
def _path_zigzag(p0, ang, n_seg, seg_m, gsd, rng):
    pts = [np.array(p0, np.float32)]
    sgn = 1
    for _ in range(n_seg):
        a = ang + sgn * np.deg2rad(rng.uniform(25, 55))
        L = rng.uniform(*seg_m) / gsd
        pts.append(pts[-1] + L * np.array([np.cos(a), np.sin(a)], np.float32))
        sgn *= -1
    return np.array(pts)


def _path_crenellated(p0, ang, n_seg, seg_m, gsd, rng):
    pts = [np.array(p0, np.float32)]
    fwd = np.array([np.cos(ang), np.sin(ang)], np.float32)
    side = np.array([-fwd[1], fwd[0]], np.float32)
    s = 1
    for i in range(n_seg):
        L = rng.uniform(*seg_m) / gsd
        pts.append(pts[-1] + fwd * L)
        if i < n_seg - 1:
            pts.append(pts[-1] + side * s * rng.uniform(1.5, 3.0) / gsd)
            s *= -1
    return np.array(pts)


def _path_wavy(p0, ang, length_m, gsd, rng):
    step = max(2.0, 1.5 / gsd)
    n = int(length_m / gsd / step)
    pts = [np.array(p0, np.float32)]
    a, da = ang, 0.0
    for _ in range(max(2, n)):
        da = 0.85 * da + rng.normal(0, 0.06)
        a += da
        pts.append(pts[-1] + step * np.array([np.cos(a), np.sin(a)], np.float32))
    return np.array(pts)


def _path_segmented(p0, ang, n_seg, seg_m, gsd, rng):
    pts = [np.array(p0, np.float32)]
    a = ang
    for _ in range(n_seg):
        a += np.deg2rad(rng.normal(0, 12))
        L = rng.uniform(*seg_m) / gsd
        pts.append(pts[-1] + L * np.array([np.cos(a), np.sin(a)], np.float32))
    return np.array(pts)


def _dist_to_polyline(shape, pts_local, thickness=1):
    canvas = np.zeros(shape, np.uint8)
    cv2.polylines(canvas, [np.round(pts_local).astype(np.int32).reshape(-1, 1, 2)], False, 255, thickness, cv2.LINE_8)
    return cv2.distanceTransform(255 - canvas, cv2.DIST_L2, 5).astype(np.float32)


def _offset_polyline(pts, off):
    out = []
    for i in range(len(pts)):
        a = pts[max(0, i - 1)]
        b = pts[min(len(pts) - 1, i + 1)]
        d = b - a
        n = np.array([-d[1], d[0]], np.float32) / (np.linalg.norm(d) + 1e-6)
        out.append(pts[i] + n * off)
    return np.array(out)


def trench_geometry(canvas, pts, rng, kind='trench', old=False):
    gsd = canvas.gsd
    if kind == 'ditch':                                       # протитанковий рів
        w_m, d_m = rng.uniform(3.0, 5.5), rng.uniform(1.8, 2.8)
        sp_w_m, sp_h_m = rng.uniform(3.0, 6.0), rng.uniform(0.8, 1.4)
        both = False
    else:                                                     # стрілецька/ходова траншея
        w_m, d_m = rng.uniform(0.8, 1.5), rng.uniform(1.1, 1.8)
        sp_w_m, sp_h_m = rng.uniform(1.0, 2.6), rng.uniform(0.25, 0.6)
        both = rng.random() < 0.55
    w, sp_w = max(1.2, w_m / gsd), max(1.5, sp_w_m / gsd)
    margin = int(np.ceil(w / 2 + sp_w * 2 + 6))
    x0 = max(0, int(np.floor(pts[:, 0].min())) - margin); x1 = min(canvas.w, int(np.ceil(pts[:, 0].max())) + margin + 1)
    y0 = max(0, int(np.floor(pts[:, 1].min())) - margin); y1 = min(canvas.h, int(np.ceil(pts[:, 1].max())) + margin + 1)
    if x1 - x0 < 4 or y1 - y0 < 4:
        return None
    loc = pts - np.array([x0, y0], np.float32)
    shape = (y1 - y0, x1 - x0)
    d = _dist_to_polyline(shape, loc)
    hh, ww = shape
    rough = fractal_noise(hh, ww, rng, sigmas=(0.8, 2.0, 5.0), weights=(0.3, 0.4, 0.3))
    wiggle = 1 + 0.12 * rough
    trench = _smoothstep(w / 2 * wiggle + 0.8, w / 2 * wiggle - 0.8, d)
    height = -d_m * trench
    side = 1 if rng.random() < 0.5 else -1
    d_front = _dist_to_polyline(shape, _offset_polyline(loc, side * (w / 2 + sp_w / 2 + 0.5)))
    berm = _smoothstep(sp_w / 2 * (1 + 0.25 * rough) + 1.0, 0, d_front) * (1 - trench)
    if both:
        d_back = _dist_to_polyline(shape, _offset_polyline(loc, -side * (w / 2 + sp_w / 3 + 0.5)))
        berm = np.maximum(berm, 0.6 * _smoothstep(sp_w / 3 * (1 + 0.25 * rough) + 1.0, 0, d_back) * (1 - trench))
    height += sp_h_m * berm * (1 + 0.2 * rough)
    patch = _smoothstep(-0.8, 0.5, fractal_noise(hh, ww, rng, sigmas=(0.8, 2.0, 6.0), weights=(0.35, 0.4, 0.25)) + 0.6)
    alpha = np.maximum(trench, berm * patch * rng.uniform(0.7, 1.0))
    dark = trench * rng.uniform(0.25, 0.5)
    if old:
        height *= rng.uniform(0.4, 0.7)
        alpha *= rng.uniform(0.25, 0.5)
        dark *= 0.4
    label = (d <= w / 2 + 1.0) | (cv2.GaussianBlur((berm * patch).astype(np.float32), (0, 0), 1.0) > 0.45)
    return dict(box=(y0, y1, x0, x1), height=height.astype(np.float32), alpha=alpha.astype(np.float32),
                dark=dark.astype(np.float32), label=label)


def dugout_geometry(canvas, cx, cy, rng, old=False):
    gsd = canvas.gsd
    side_m, d_m = rng.uniform(2.0, 4.0), rng.uniform(1.2, 2.0)
    s = side_m / gsd
    sp = rng.uniform(1.0, 2.0) / gsd
    R = int(np.ceil(s + 2 * sp + 4))
    x0, x1 = max(0, int(cx) - R), min(canvas.w, int(cx) + R + 1)
    y0, y1 = max(0, int(cy) - R), min(canvas.h, int(cy) + R + 1)
    if x1 - x0 < 4 or y1 - y0 < 4:
        return None
    yy, xx = np.mgrid[y0:y1, x0:x1].astype(np.float32)
    th = rng.uniform(0, np.pi / 2)
    u = (xx - cx) * np.cos(th) + (yy - cy) * np.sin(th)
    v = -(xx - cx) * np.sin(th) + (yy - cy) * np.cos(th)
    dbox = np.maximum(np.abs(u), np.abs(v))                   # «квадратна» відстань
    pit = _smoothstep(s / 2 + 0.8, s / 2 - 0.8, dbox)
    ring = _smoothstep(s / 2 + sp + 1, s / 2 + sp * 0.3, dbox) * (1 - pit)
    hh, ww = dbox.shape
    patch = _smoothstep(-0.8, 0.5, fractal_noise(hh, ww, rng, sigmas=(0.8, 2.0), weights=(0.5, 0.5)) + 0.6)
    height = -d_m * pit + rng.uniform(0.3, 0.6) * ring
    alpha = np.maximum(pit, ring * patch)
    dark = pit * rng.uniform(0.3, 0.6)
    if old:
        height *= 0.5; alpha *= 0.4; dark *= 0.4
    label = (dbox <= s / 2 + 1) | ((ring * patch) > 0.5)
    return dict(box=(y0, y1, x0, x1), height=height.astype(np.float32), alpha=alpha.astype(np.float32),
                dark=dark.astype(np.float32), label=label)


def dragon_teeth_geometry(canvas, p0, ang, rng):
    """«Зуби дракона» — ряди бетонних пірамід (світлі, з тінями)."""
    gsd = canvas.gsd
    size_m, h_m = rng.uniform(1.0, 1.5), rng.uniform(0.9, 1.3)
    spacing = rng.uniform(1.9, 2.6) / gsd
    rows, n = int(rng.integers(2, 5)), int(rng.integers(10, 35))
    fwd = np.array([np.cos(ang), np.sin(ang)], np.float32)
    side = np.array([-fwd[1], fwd[0]], np.float32)
    centers = [np.asarray(p0, np.float32) + fwd * spacing * (i + 0.5 * (r % 2)) + side * spacing * r * 0.9
               for r in range(rows) for i in range(n)]
    centers = np.array(centers)
    s = size_m / gsd
    margin = int(np.ceil(s + 4))
    x0 = max(0, int(centers[:, 0].min()) - margin); x1 = min(canvas.w, int(centers[:, 0].max()) + margin + 1)
    y0 = max(0, int(centers[:, 1].min()) - margin); y1 = min(canvas.h, int(centers[:, 1].max()) + margin + 1)
    if x1 - x0 < 4 or y1 - y0 < 4:
        return None
    yy, xx = np.mgrid[y0:y1, x0:x1].astype(np.float32)
    height = np.zeros(yy.shape, np.float32)
    for c in centers:
        u = np.abs(xx - c[0]); v = np.abs(yy - c[1])
        pyr = np.clip(1 - np.maximum(u, v) / (s / 2 + 1e-6), 0, 1)
        height = np.maximum(height, pyr * h_m)
    body = (height > 0.02).astype(np.float32)
    alpha = body
    label = cv2.dilate(body.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    return dict(box=(y0, y1, x0, x1), height=height, alpha=alpha, dark=np.zeros_like(alpha), label=label,
                color=np.array([175, 172, 165], np.float32))


# ------------------------------------------------------------------ сцена
def _place(canvas, geom, soil, label_map=None, lbl=None, aux=None, aux_val=None, protect=None):
    if geom is None:
        return 0
    y0, y1, x0, x1 = geom['box']
    canvas.add(y0, y1, x0, x1, geom['height'], geom['alpha'], geom.get('color', soil), geom['dark'])
    lab = geom['label']
    if label_map is not None:
        region = label_map[y0:y1, x0:x1]
        ok = lab.copy()
        if protect is not None:
            ok &= ~protect[y0:y1, x0:x1]
        region[ok] = lbl
        return int(ok.sum())
    if aux is not None:
        aux[y0:y1, x0:x1][lab] = aux_val
    return 0


def _trench_path(canvas, openness, rng, regime, gsd):
    for _ in range(12):
        p0 = sample_point(openness, rng, margin=10)
        if p0 is None:
            return None, None
        ang = rng.uniform(0, 2 * np.pi)
        r = rng.random()
        if r < 0.12 * (regime['p_ditch'] / 0.12):
            kind = 'ditch'
            pts = _path_segmented(p0, ang, int(rng.integers(2, 5)), (25, 60), gsd, rng)
        else:
            kind = 'trench'
            style = rng.choice(regime['trench_styles'])
            if style == 'zigzag':
                pts = _path_zigzag(p0, ang, int(rng.integers(4, 14)), (3, 8), gsd, rng)
            elif style == 'crenellated':
                pts = _path_crenellated(p0, ang, int(rng.integers(3, 9)), (4, 9), gsd, rng)
            elif style == 'wavy':
                pts = _path_wavy(p0, ang, rng.uniform(25, 110), gsd, rng)
            else:
                pts = _path_segmented(p0, ang, int(rng.integers(2, 7)), (6, 20), gsd, rng)
        pts[:, 0] = np.clip(pts[:, 0], 2, canvas.w - 3)
        pts[:, 1] = np.clip(pts[:, 1], 2, canvas.h - 3)
        # перевірка: траса має пролягати переважно відкритою місцевістю
        dense = np.concatenate([np.linspace(pts[i], pts[i + 1], max(2, int(np.linalg.norm(pts[i + 1] - pts[i]) // 2)))
                                for i in range(len(pts) - 1)])
        xi = np.clip(dense[:, 0].astype(int), 0, canvas.w - 1); yi = np.clip(dense[:, 1].astype(int), 0, canvas.h - 1)
        if (openness[yi, xi] > 0.12).mean() >= 0.85 and np.ptp(pts[:, 0]) + np.ptp(pts[:, 1]) > 8:
            return pts, kind
    return None, None


def _visible_label(label, before, after, classes=(LBL_CRATER, LBL_FORT), thr=9.0, min_px=6):
    """Мітка лише там, де зміна реально видима: |after-before| у RGB > thr (з невеликим допуском).
    Об'єкти, які після рендеру майже не видно, повністю знімаються з розмітки."""
    diff = np.sqrt(((after.astype(np.float32) - before.astype(np.float32)) ** 2).sum(2))
    vis = cv2.GaussianBlur(diff, (0, 0), 1.0) > thr
    vis = cv2.dilate(vis.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    out = label.copy()
    for c in classes:
        m = label == c
        if not m.any():
            continue
        keep = m & vis
        keep = cv2.morphologyEx(keep.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8)).astype(bool) & m
        n, comp, stats, _ = cv2.connectedComponentsWithStats(keep.astype(np.uint8), connectivity=8)
        small = np.isin(comp, np.where(stats[:, cv2.CC_STAT_AREA] < min_px)[0])
        keep &= ~small
        out[m & ~keep] = LBL_BG
    return out


def synthesize_pair(pre, post, gsd, rng, regime='id', base_label=None, exclusion=None, sun=None,
                    p_crater=0.75, p_fort=0.6, p_old=0.45, p_empty=0.12):
    """Повертає (pre_out, post_out, label, aux, info).
    label: 0 фон / 2 нова вирва / 3 свіжа фортифікація (+ збережені мітки бази: 1/4/254/255)
    aux:   1 «стара» вирва, 2 «стара» фортифікація (є на ОБОХ знімках, мітка 0) — для shortcut-тесту."""
    R = SYN_REGIMES[regime]
    palette = SOIL_PALETTES[R['palette']]
    h, w = post.shape[:2]
    label = np.zeros((h, w), np.uint8) if base_label is None else base_label.copy()
    protect = label != 0                                        # не перезаписуємо мітки бази
    aux = np.zeros((h, w), np.uint8)
    excl = protect.copy() if exclusion is None else (protect | exclusion)
    st_post, st_pre = scene_stats(post), scene_stats(pre)
    nat_post, any_post = ground_maps(post, excl)
    az = rng.uniform(0, 360)
    el_post = rng.uniform(*R['sun_elev']); el_pre = rng.uniform(*R['sun_elev'])
    if sun is not None and regime == 'id':          # реальне сонце знімка «після» (метадані xBD)
        az, el_post = float(sun[0]) % 360, float(np.clip(sun[1], 15, 75))
    az_pre = (az + rng.normal(0, 25)) % 360
    pal_name = rng.choice(list(palette))
    cv_post = Canvas(post, gsd, az, el_post, rng, palette[pal_name], st_post['lum'])
    cv_pre = Canvas(pre, gsd, az_pre, el_pre, rng, palette[pal_name], st_pre['lum'])
    info = {'n_crater': 0, 'n_fort': 0, 'n_old': 0, 'palette': str(pal_name), 'regime': regime}

    def soil_color(cv):
        name = rng.choice(list(palette))
        ref = np.array(palette[pal_name], np.float32)
        scale = cv.soil / (ref + 1e-6)
        col = np.array(palette[name], np.float32) * scale * rng.uniform(0.92, 1.08, 3)
        grey = col.mean()
        return grey + (col - grey) * rng.uniform(0.55, 0.9)        # приглушена насиченість

    def crater_diam():
        mix = R['crater_diam_mix']
        k = rng.choice(len(mix), p=np.array([m[2] for m in mix]) / sum(m[2] for m in mix))
        lo, hi, _ = mix[k]
        return float(np.exp(rng.uniform(np.log(lo), np.log(hi))))

    # --- «старі» об'єкти на ОБОХ знімках (мітка 0, aux) — однакова геометрія, різний рендер
    if rng.random() < p_old:
        seed_geo = int(rng.integers(1 << 30))
        for _ in range(int(rng.integers(1, 6))):
            p = sample_point(any_post, rng)
            if p is None:
                break
            d = crater_diam()
            sc = soil_color(cv_post)
            g_post = crater_geometry(cv_post, p[0], p[1], d, np.random.default_rng(seed_geo), sc, old=True)
            g_pre = crater_geometry(cv_pre, p[0], p[1], d, np.random.default_rng(seed_geo), sc, old=True)
            seed_geo += 1
            _place(cv_post, g_post, sc, aux=aux, aux_val=1)
            _place(cv_pre, g_pre, sc)
            info['n_old'] += 1
        if rng.random() < 0.4:
            pts, kind = _trench_path(cv_post, nat_post, rng, R, gsd)
            if pts is not None:
                g_post = trench_geometry(cv_post, pts, np.random.default_rng(seed_geo), kind, old=True)
                g_pre = trench_geometry(cv_pre, pts, np.random.default_rng(seed_geo), kind, old=True)
                sc = soil_color(cv_post)
                _place(cv_post, g_post, sc, aux=aux, aux_val=2)
                _place(cv_pre, g_pre, sc)
                info['n_old'] += 1
    old_mask = aux > 0
    free = ~cv2.dilate(old_mask.astype(np.uint8), np.ones((9, 9), np.uint8)).astype(bool)
    nat_new, any_new = nat_post * free, any_post * free

    empty = rng.random() < p_empty
    if not empty:
        want_crater = rng.random() < p_crater
        want_fort = rng.random() < p_fort
        if not (want_crater or want_fort):
            want_crater = rng.random() < 0.5
            want_fort = not want_crater
        if want_crater:
            for _ in range(int(rng.integers(1, 4))):
                c = sample_point(any_new, rng)
                if c is None:
                    break
                spread = rng.uniform(4, 35) / gsd
                for _ in range(int(rng.integers(1, 20))):
                    if rng.random() < 0.15:                      # частина — будь-де (дороги, подвір'я)
                        q = (rng.uniform(10, w - 10), rng.uniform(10, h - 10))
                    else:
                        q = (c[0] + rng.normal(0, spread), c[1] + rng.normal(0, spread))
                    if not (4 <= q[0] < w - 4 and 4 <= q[1] < h - 4):
                        continue
                    # приймаємо точку з імовірністю ~ «природності» місця (дахи/виключені зони — майже ніколи)
                    if rng.random() > min(1.0, any_new[int(q[1]), int(q[0])] / 0.35):
                        continue
                    d = crater_diam()
                    if d / 2 / gsd < 1.2:                        # надто мала для розмітки
                        continue
                    sc = soil_color(cv_post)
                    g = crater_geometry(cv_post, q[0], q[1], d, rng, sc)
                    if _place(cv_post, g, sc, label_map=label, lbl=LBL_CRATER, protect=protect) > 0:
                        info['n_crater'] += 1
        if want_fort:
            for _ in range(int(rng.integers(1, 4))):
                pts, kind = _trench_path(cv_post, nat_new, rng, R, gsd)
                if pts is None:
                    break
                g = trench_geometry(cv_post, pts, rng, kind)
                if _place(cv_post, g, soil_color(cv_post), label_map=label, lbl=LBL_FORT, protect=protect) > 0:
                    info['n_fort'] += 1
                if kind == 'trench' and rng.random() < 0.5:     # бліндажі/позиції біля траншеї
                    for _ in range(int(rng.integers(1, 4))):
                        i = int(rng.integers(0, len(pts)))
                        off = rng.normal(0, 4 / gsd, 2)
                        g = dugout_geometry(cv_post, pts[i, 0] + off[0], pts[i, 1] + off[1], rng)
                        _place(cv_post, g, soil_color(cv_post), label_map=label, lbl=LBL_FORT, protect=protect)
            if rng.random() < R['p_dragon']:
                p = sample_point(nat_new, rng)
                if p is not None:
                    g = dragon_teeth_geometry(cv_post, p, rng.uniform(0, 2 * np.pi), rng)
                    if _place(cv_post, g, None, label_map=label, lbl=LBL_FORT, protect=protect) > 0:
                        info['n_fort'] += 1
    blur = rng.uniform(0.35, 0.8)
    post_out = cv_post.render(st_post['noise'], blur)
    pre_out = cv_pre.render(st_pre['noise'], blur) if info['n_old'] else pre.copy()
    label = _visible_label(label, post, post_out)
    info['px_crater'] = int((label == LBL_CRATER).sum())
    info['px_fort'] = int((label == LBL_FORT).sum())
    return pre_out, post_out, label, aux, info

In [ ]:
# ==============================================================================
# 5. Приклади для інтерфейсу — ЛИШЕ з теки demo_pairs на Google Drive (тільки читання)
# ==============================================================================
# Пара: <ім'я>_pre + <ім'я>_post  або  <ім'я>_before + <ім'я>_after  (також просто before.png + after.png).
# Розмітка (необов'язково): <ім'я>_label.png поруч (для before/after без імені — label.png).
# Шукаємо в demo_pairs і її підтеках, крім службових тек, що починаються з «_» (напр. _old).
# Ноутбук у demo_pairs нічого не записує й нових прикладів не формує.
IMG_EXT = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp', '.webp')
_SIDE_RE = re.compile(r'^(?:(?P<base>.*)[_\-\s])?(?P<tag>pre|before|post|after)$', re.IGNORECASE)
LABEL_FOR = {}                 # ім'я файлу «Після» → шлях до розмітки (Gradio передає копію файлу з тим самим ім'ям)


def pairs_from_demo_dir(root=DEMO_PAIRS):
    LABEL_FOR.clear()
    root = Path(root)
    if not root.is_dir():
        return []
    groups = {}
    for p in sorted(root.rglob('*')):
        rel = p.relative_to(root)
        if not p.is_file() or p.suffix.lower() not in IMG_EXT or any(s.startswith(('_', '.')) for s in rel.parts[:-1]):
            continue
        m = _SIDE_RE.match(p.stem)
        if m is None:
            continue
        side = 'pre' if m['tag'].lower() in ('pre', 'before') else 'post'
        groups.setdefault((p.parent, m['base'] or ''), {}).setdefault(side, p)
    out = []
    for (folder, base), g in sorted(groups.items(), key=lambda kv: (str(kv[0][0]), kv[0][1])):
        name = (folder / (base or '(before/after)')).relative_to(root).as_posix()
        if 'pre' not in g or 'post' not in g:
            print(f'  ⚠ {name}: немає пари', '«До»' if 'pre' not in g else '«Після»')
            continue
        lab = folder / (f'{base}_label.png' if base else 'label.png')
        if lab.exists():
            LABEL_FOR[g['post'].name] = lab
        out.append([str(g['pre']), str(g['post'])])
    return out


def gt_path_for(post_path):
    """Розмітка для прикладу з demo_pairs (за ім'ям файлу «Після»); для власних знімків — None."""
    if not post_path:
        return None
    return LABEL_FOR.get(Path(str(post_path)).name)


EXAMPLES = pairs_from_demo_dir()
if EXAMPLES:
    print(f'Прикладів з {DEMO_PAIRS}: {len(EXAMPLES)} (з розміткою: {len(LABEL_FOR)})')
    for pre, post in EXAMPLES:
        print(f'  • {Path(pre).relative_to(DEMO_PAIRS).as_posix()}  +  {Path(post).name}',
              '+ GT' if Path(post).name in LABEL_FOR else '')
else:
    print(f'⚠ У {DEMO_PAIRS} не знайдено пар «До»/«Після» — інтерфейс працюватиме без прикладів (завантажуйте власні знімки).')

In [ ]:
# ==============================================================================
# 6. Ядро демо: файли «До»/«Після» → зміни, об'єкти, візуалізації, файли результату
# ==============================================================================
def fit(img, max_side=1400):
    """Зменшення лише для показу в інтерфейсі (файли результату — у повній роздільності)."""
    if img is None:
        return None
    h, w = img.shape[:2]
    s = max_side / max(h, w)
    if s >= 1:
        return img
    interp = cv2.INTER_NEAREST if img.ndim == 2 or img.dtype != np.uint8 else cv2.INTER_AREA
    return cv2.resize(img, (int(round(w * s)), int(round(h * s))), interpolation=interp)


def _font(size):
    try:
        return ImageFont.truetype(FONT_PATH, size)
    except Exception:
        return ImageFont.load_default()


def draw_boxes(img, objects, max_labels=40):
    """Рамки для всіх об'єктів; підписи — лише для max_labels найвпевненіших (щоб не захаращувати)."""
    im = Image.fromarray(img)
    d = ImageDraw.Draw(im)
    side = min(img.shape[:2])
    font = _font(max(12, side // 64))
    lw = max(2, side // 450)
    for k, o in enumerate(sorted(objects, key=lambda o: -o['score'])):
        x, y, w, h = o['bbox_xywh']
        col = CLASS_COLORS[o['cls']]
        d.rectangle([x, y, x + w - 1, y + h - 1], outline=col, width=lw)
        if k >= max_labels:
            continue
        txt = f"#{o['id']} {CLASS_UA_SHORT[o['cls']]} {o['score']:.2f}"
        l, t, r, b = d.textbbox((0, 0), txt, font=font)
        tw, th = r - l, b - t
        ty = y - th - 6 if y - th - 6 >= 0 else min(y + h + 2, img.shape[0] - th - 6)
        d.rectangle([x, ty, x + tw + 6, ty + th + 6], fill=col)
        d.text((x + 3, ty + 2 - t), txt, fill=(0, 0, 0) if sum(col) > 400 else (255, 255, 255), font=font)
    return np.array(im)


def render_mask(cls_map, shown):
    out = np.zeros(cls_map.shape + (3,), np.uint8)
    for c in shown:
        out[cls_map == c] = CLASS_COLORS[c]
    return out


def render_prob(prob, post):
    heat = cv2.applyColorMap((np.clip(prob, 0, 1) * 255).astype(np.uint8), cv2.COLORMAP_INFERNO)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    return (0.75 * heat + 0.25 * post).astype(np.uint8)


def render_gt(post, lab, aux=None):
    out = label_overlay(post, lab)
    m = lab == LBL_IGNORE
    out[m] = (out[m] * 0.6 + 128 * 0.4).astype(np.uint8)
    if aux is not None and aux.any():
        m = aux > 0
        out[m] = (out[m] * 0.55 + np.array([0, 220, 255]) * 0.45).astype(np.uint8)
    return out


def frame_scores(cls_map, lab, classes=(1, 2, 3, 4), binary=True):
    """F1 кадру проти розмітки: бінарний (будь-яка зміна) і по класах, що є в GT або в прогнозі."""
    valid = lab != LBL_IGNORE
    gt_chg = (((lab >= 1) & (lab <= 4)) | (lab == LBL_UNKNOWN)) & valid
    pr_chg = (cls_map > 0) & valid

    def f1(p, g):
        tp, fp, fn = int((p & g).sum()), int((p & ~g).sum()), int((~p & g).sum())
        return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else float('nan')
    res = {'будь-яка зміна': f1(pr_chg, gt_chg)} if binary else {}
    for c in classes:
        g, p = (lab == c) & valid, (cls_map == c) & valid
        if g.any() or p.any():
            res[CLASS_NAMES[c]] = f1(p, g)
    return res


def run_detection(pre_path, post_path, thr=None, gsd=None, tta=False, synthetic=False, syn_seed=0,
                  autoresize=False, label_path=None):
    t0 = time.perf_counter()
    pre, post = read_rgb(pre_path), read_rgb(post_path)
    notes = []
    if pre.shape != post.shape:
        if not autoresize:
            raise ValueError(f'Знімки мають бути вирівняні й однакового розміру: «До» {pre.shape[1]}×{pre.shape[0]}, '
                             f'«Після» {post.shape[1]}×{post.shape[0]}. Увімкніть «Підігнати розмір», якщо це той самий кадр.')
        post = cv2.resize(post, (pre.shape[1], pre.shape[0]), interpolation=cv2.INTER_AREA)
        notes.append('розмір «Після» підігнано під «До»')
    t_read = time.perf_counter() - t0
    gt = read_gray(label_path) if label_path else None
    if gt is not None and gt.shape != pre.shape[:2]:
        gt = None
        notes.append('розмітку прикладу не використано: її розмір не збігається з парою')
    syn = None
    if synthetic:
        g = gsd or DEFAULT_GSD_UNKNOWN
        ts = time.perf_counter()
        pre, post, syn_lab, syn_aux, info = synthesize_pair(pre, post, g, np.random.default_rng(int(syn_seed)), 'id',
                                                            None, None, None, p_empty=0.0, p_old=0.6)
        syn = dict(label=syn_lab, aux=syn_aux, info=info, gen_s=time.perf_counter() - ts)
        gt = syn_lab
        notes.append(f'🧪 вставлено синтетику (GSD {g} м/px): вирв {info["n_crater"]}, фортифікацій {info["n_fort"]}, '
                     f'«старих» об\'єктів на обох знімках {info["n_old"]}')
    cuda_sync(); t1 = time.perf_counter()
    prob, typ, conf = predict_full(MODEL, pre, post, stride=STRIDE, tta=tta, return_type_prob=True)
    cuda_sync(); t2 = time.perf_counter()
    thr = THRESHOLD if thr is None else float(thr)
    cls_map = pred_class_map(prob, typ, thr)
    objects = extract_objects(cls_map, prob, conf, gsd)
    t3 = time.perf_counter()
    timing = dict(read_s=t_read, inference_s=t2 - t1, postprocess_s=t3 - t2, total_s=t_read + (t3 - t1))
    return dict(pre=pre, post=post, prob=prob, cls_map=cls_map, objects=objects, timing=timing, thr=thr, gsd=gsd,
                tta=tta, gt=gt, syn=syn, notes=notes,
                scores=(frame_scores(cls_map, gt, (LBL_CRATER, LBL_FORT), binary=False) if synthetic
                        else frame_scores(cls_map, gt) if gt is not None else None))


def save_outputs(res, stem='result'):
    d = OUT_DIR / f'{time.strftime("%Y%m%d_%H%M%S")}_{stem}'[:120]
    d.mkdir(parents=True, exist_ok=True)
    paths = {'overlay': d / 'overlay_bbox.png', 'mask_color': d / 'mask_classes_color.png',
             'mask_index': d / 'mask_classes_index.png', 'prob': d / 'change_probability.png', 'json': d / 'result.json'}
    cv2.imwrite(str(paths['overlay']), cv2.cvtColor(draw_boxes(label_overlay(res['post'], res['cls_map']), res['objects']), cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(paths['mask_color']), cv2.cvtColor(render_mask(res['cls_map'], (1, 2, 3, 4)), cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(paths['mask_index']), res['cls_map'])          # 0 — без змін, 1..4 — класи
    cv2.imwrite(str(paths['prob']), (np.clip(res['prob'], 0, 1) * 255).astype(np.uint8))
    paths['json'].write_text(json.dumps(dict(
        classes={str(k): v for k, v in CLASS_NAMES.items()}, threshold=res['thr'], gsd_m_per_px=res['gsd'], tta=res['tta'],
        image_size=list(res['cls_map'].shape[::-1]), timing_s=res['timing'], device=DEVICE_NAME, notes=res['notes'],
        frame_f1=res['scores'], objects=res['objects']), ensure_ascii=False, indent=1, default=float))
    return [str(p) for p in paths.values()]


def show_result(res, title=''):
    """Запасний показ без Gradio (matplotlib)."""
    panels = [(res['pre'], 'До'), (res['post'], 'Після'),
              (draw_boxes(label_overlay(res['post'], res['cls_map']), res['objects']), f'Зміни: {len(res["objects"])} об\'єктів'),
              (render_mask(res['cls_map'], (1, 2, 3, 4)), 'Маска класів')]
    if res['gt'] is not None:
        panels.append((render_gt(res['post'], res['gt'], res['syn']['aux'] if res['syn'] else None), 'Розмітка'))
    fig, ax = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5.4))
    for a, (im, t) in zip(ax, panels):
        a.imshow(im); a.set_title(t); a.axis('off')
    t = res['timing']
    fig.suptitle(f"{title} | інференс {t['inference_s']:.3f} с, разом {t['total_s']:.3f} с ({DEVICE_NAME})")
    plt.tight_layout(); plt.show()


# швидка перевірка на першому прикладі (якщо є)
if EXAMPLES:
    _r = run_detection(*EXAMPLES[0], gsd=None, label_path=gt_path_for(EXAMPLES[0][1]))
    show_result(_r, Path(EXAMPLES[0][1]).name)
    print('Об\'єктів:', Counter(o['name'] for o in _r['objects']), '| час:', {k: round(v, 3) for k, v in _r['timing'].items()})

In [ ]:
# ==============================================================================
# 7. Інтерфейс Gradio
# ==============================================================================
CLASS_LABELS = {c: f'{c}: {CLASS_NAMES[c]}' for c in (1, 2, 3, 4)}
TABLE_COLS = ['#', 'клас', 'площа, px', 'площа, м²', 'рамка (x, y, w, h)', 'впевненість']
LEGEND = ' &nbsp; &nbsp; '.join(f'<span style="display:inline-block;width:16px;height:16px;background:rgb{CLASS_COLORS[c]};'
                                 f'border:1px solid #444;vertical-align:-2px"></span> {CLASS_NAMES[c]}' for c in (1, 2, 3, 4))
# більший шрифт заголовка, інструкції, легенди та підсумку
UI_CSS = '''
.eps-title h1, .eps-title h2 {font-size: 2.1rem !important; line-height: 1.25 !important}
.eps-big, .eps-big p, .eps-big li, .eps-big span, .eps-big strong, .eps-big em {font-size: 1.2rem !important; line-height: 1.6 !important}
'''
BENCH_HINT = ('*⏱ Замір швидкодії вимірює лише час (вимога ≤ 2 с на кадр 1024×1024): 3 прогрівальні + 20 вимірюваних прогонів '
              'моделі з постобробкою. Якщо пару завантажено — вона масштабується до 1024×1024, інакше береться випадковий кадр. '
              'Маски й рамки не змінюються; результат з\'явиться тут (≈ 5–15 с на T4).*')


def ui_preview(f):
    if not f:
        return None
    try:
        return fit(read_rgb(f), 900)
    except Exception:
        return None


def _summary_md(res, shown_objs):
    t = res['timing']
    cnt = Counter(o['cls'] for o in shown_objs)
    h, w = res['cls_map'].shape
    lines = ['**Знайдено:** ' + ' · '.join(f'{CLASS_NAMES[c]}: **{cnt.get(c, 0)}**' for c in (1, 2, 3, 4)),
             f"**Час обробки:** читання {t['read_s']:.3f} с · інференс {t['inference_s']:.3f} с · "
             f"постобробка {t['postprocess_s']:.3f} с · **разом {t['total_s']:.3f} с** ({DEVICE_NAME})"
             + (' — ✅ ≤ 2 с' if t['total_s'] <= 2 else ' — ⚠ > 2 с'),
             f"**Кадр:** {w}×{h} px · поріг {res['thr']:.2f} · вікно {TILE}/крок {STRIDE} · TTA {'так' if res['tta'] else 'ні'}"
             + (f" · GSD {res['gsd']} м/px" if res['gsd'] else ' · GSD невідомий (площа лише в px)')]
    lines += [f'*{n}*' for n in res['notes']]
    if res['scores']:
        src = 'вставленої синтетики' if res['syn'] else 'розмітки прикладу'
        lines.append(f'**F1 цього кадру (проти {src}):** ' +
                     ' · '.join(f'{k}: {v:.2f}' for k, v in res['scores'].items() if v == v))
    return '\n\n'.join(lines)


def ui_process(pre_f, post_f, thr, gsd, tta, synthetic, seed, classes, min_score, autoresize):
    if not pre_f or not post_f:
        raise gr.Error('Завантажте обидва знімки: «До» і «Після».')
    shown = [c for c, lbl in CLASS_LABELS.items() if lbl in (classes or [])]
    try:
        res = run_detection(pre_f, post_f, thr=thr, gsd=float(gsd) if gsd and gsd > 0 else None, tta=bool(tta),
                            synthetic=bool(synthetic), syn_seed=int(seed), autoresize=bool(autoresize),
                            label_path=None if synthetic else gt_path_for(post_f))
    except ValueError as e:
        raise gr.Error(str(e))
    objs = [o for o in res['objects'] if o['cls'] in shown and o['score'] >= float(min_score)]
    cls_shown = np.where(np.isin(res['cls_map'], shown), res['cls_map'], 0).astype(np.uint8)
    overlay = draw_boxes(label_overlay(res['post'], cls_shown), objs)
    mask = render_mask(res['cls_map'], shown)
    prob = render_prob(res['prob'], res['post'])
    gt_img = None
    if res['gt'] is not None:
        gt_img = fit(render_gt(res['post'], res['gt'], res['syn']['aux'] if res['syn'] else None))
    table = pd.DataFrame([[o['id'], o['name'], o['area_px'], o['area_m2'] if o['area_m2'] is not None else '—',
                           str(o['bbox_xywh']), o['score']] for o in objs], columns=TABLE_COLS)
    files = save_outputs(res, Path(str(post_f)).stem)
    return (fit(res['pre'], 900), fit(res['post'], 900), fit(overlay), fit(mask), fit(prob), gt_img, table,
            _summary_md(res, objs), files)


BENCH_RUNS = [0]                                                  # лічильник: видно, що замір оновився


def ui_benchmark(pre_f, post_f, tta, n=20, progress=gr.Progress()):
    if pre_f and post_f:
        a = cv2.resize(read_rgb(pre_f), (1024, 1024), interpolation=cv2.INTER_AREA)
        b = cv2.resize(read_rgb(post_f), (1024, 1024), interpolation=cv2.INTER_AREA)
        src = 'завантажена пара, приведена до 1024×1024'
    else:
        rng = np.random.default_rng(0)
        a = rng.integers(0, 255, (1024, 1024, 3), dtype=np.uint8); b = a.copy()
        src = 'випадковий кадр 1024×1024 (знімки не завантажено)'
    progress(0, desc='Прогрів моделі…')
    for _ in range(3):                                            # прогрів
        predict_full(MODEL, a, b, tta=bool(tta))
    times = []
    for i in range(n):
        progress(i / n, desc=f'Прогін {i + 1}/{n}')
        cuda_sync(); t0 = time.perf_counter()
        prob, typ, conf = predict_full(MODEL, a, b, tta=bool(tta), return_type_prob=True)
        extract_objects(pred_class_map(prob, typ, THRESHOLD), prob, conf, None)
        cuda_sync(); times.append(time.perf_counter() - t0)
    t = np.array(times)
    ok = '✅ вимогу ≤ 2 с виконано' if t.max() <= 2 else '⚠ вимогу ≤ 2 с НЕ виконано'
    BENCH_RUNS[0] += 1
    return (f'**Замір №{BENCH_RUNS[0]} ({DEVICE_NAME}, {src}, TTA {"так" if tta else "ні"}, {n} прогонів):** '
            f'середнє **{t.mean():.3f} с** · медіана {np.median(t):.3f} с · p95 {np.percentile(t, 95):.3f} с · '
            f'максимум {t.max():.3f} с — {ok}. Вимірюється повний конвеєр від масивів у пам\'яті до списку об\'єктів.')


import inspect
_CSS_IN_LAUNCH = 'css' in inspect.signature(gr.Blocks.launch).parameters    # Gradio 6: css → launch(); Gradio 4–5: Blocks(css=…)
with gr.Blocks(title='Change Detection · кейс 4.2', **({} if _CSS_IN_LAUNCH else {'css': UI_CSS})) as demo:
    gr.Markdown('## 🛰 Детекція змін на вирівняних аерознімках · кейс 4.2', elem_classes='eps-title')
    gr.Markdown('Завантажте **«До»** і **«Після»** (PNG/JPG/TIFF, однаковий розмір, вирівняні) → **🔍 Знайти зміни**.',
                elem_classes='eps-big')
    with gr.Row():
        with gr.Column(scale=1):
            pre_f = gr.File(label='Знімок «До»', file_types=['image', '.tif', '.tiff'], type='filepath')
            pre_v = gr.Image(label='«До»', interactive=False, height=300)
        with gr.Column(scale=1):
            post_f = gr.File(label='Знімок «Після»', file_types=['image', '.tif', '.tiff'], type='filepath')
            post_v = gr.Image(label='«Після»', interactive=False, height=300)
        with gr.Column(scale=1):
            run_btn = gr.Button('🔍 Знайти зміни', variant='primary')
            thr_in = gr.Slider(0.05, 0.95, value=round(THRESHOLD, 2), step=0.01,
                               label='Поріг ймовірності зміни (за замовчуванням — обраний на validation)')
            score_in = gr.Slider(0.0, 1.0, value=0.0, step=0.05, label="Мін. впевненість об'єкта (для рамок і таблиці)")
            gsd_in = gr.Number(value=0, label='GSD, м/px (0 = невідомо; потрібен для площі в м² і масштабу синтетики)')
            cls_in = gr.CheckboxGroup(choices=list(CLASS_LABELS.values()), value=list(CLASS_LABELS.values()),
                                      label='Показувати класи')
            tta_in = gr.Checkbox(value=False, label='TTA ×4 (віддзеркалення; трохи точніше, ×4 довше)')
            resize_in = gr.Checkbox(value=False, label='Підігнати розмір «Після» під «До», якщо відрізняється')
            syn_in = gr.Checkbox(value=False, label='🧪 Вставити синтетичні вирви та траншеї у «Після» (демо класів 2–3)')
            seed_in = gr.Slider(0, 999, value=7, step=1, label='Seed синтетики')
            bench_btn = gr.Button('⏱ Замір швидкодії 1024×1024 (20 прогонів)')
            bench_md = gr.Markdown(BENCH_HINT)
    summary = gr.Markdown(elem_classes='eps-big')
    gr.Markdown(LEGEND, elem_classes='eps-big')
    with gr.Row():
        out_overlay = gr.Image(label='«Після» + рамки виявлених змін (#, клас, впевненість)', interactive=False)
        out_mask = gr.Image(label='Маска змін (колір = клас)', interactive=False)
    with gr.Row():
        out_prob = gr.Image(label='Ймовірність зміни', interactive=False)
        out_gt = gr.Image(label='Розмітка (вставлена синтетика або GT прикладу); блакитне — «старі» об\'єкти на обох знімках',
                          interactive=False)
    out_table = gr.Dataframe(headers=TABLE_COLS, label="Виявлені об'єкти", interactive=False, wrap=True)
    out_files = gr.File(label='Результат: оверлей, маски (колір та індекси 0–4), ймовірність, JSON', file_count='multiple')
    if EXAMPLES:
        gr.Examples(examples=EXAMPLES, inputs=[pre_f, post_f], label='Приклади: клікніть, потім «🔍 Знайти зміни»')
    pre_f.change(ui_preview, inputs=pre_f, outputs=pre_v)
    post_f.change(ui_preview, inputs=post_f, outputs=post_v)
    run_btn.click(ui_process,
                  inputs=[pre_f, post_f, thr_in, gsd_in, tta_in, syn_in, seed_in, cls_in, score_in, resize_in],
                  outputs=[pre_v, post_v, out_overlay, out_mask, out_prob, out_gt, out_table, summary, out_files])
    bench_btn.click(ui_benchmark, inputs=[pre_f, post_f, tta_in], outputs=bench_md)

demo.queue(max_size=8).launch(share=True, debug=False, show_error=True,
                              allowed_paths=[str(DEMO_PAIRS), str(WORK), str(OUT_DIR)],
                              **({'css': UI_CSS} if _CSS_IN_LAUNCH else {}))

## Як провести демонстрацію (сценарій з листочка)

| Пункт | Що робити в інтерфейсі |
|---|---|
| 1. Пара знімків «До»/«Після» | Приклад або власні файли — прев'ю з'являться під полями завантаження |
| 2. Запуск обробки | **🔍 Знайти зміни** |
| 3. Знайдені зміни | «Після» з рамками, маска класів, карта ймовірності, таблиця об'єктів |
| 4. Вирви / зруйновані об'єкти / фортифікації | зруйновані — приклади xBD та xBD-Ukraine; вирви й фортифікації — увімкніть **🧪** на будь-якій реальній парі (поруч з'явиться розмітка вставлених об'єктів; блакитні «старі» об'єкти є на обох знімках і **не** мають позначатися як зміна) |
| 5. Час ≤ 2 с на 1024×1024 | рядок «Час обробки» після кожного запуску + кнопка **⏱ Замір швидкодії** |
| 6. F1 ≥ 0,75 | таблиці з `REPORT.md` основного ноутбука; для прикладів з розміткою інтерфейс показує F1 конкретного кадру |
| 7. Підхід і обмеження | розділ 14.2 основного ноутбука |

**Поради:** повзунок «Мін. впевненість об'єкта» ≈ 0,3–0,4 прибирає слабкі хибні рамки; GSD (м/px) задавайте, якщо відомий — тоді площа в м² і масштаб синтетики будуть коректні (FOTBCD — 0,2; xBD — ≈0,3–0,5).

**Приклади** беруться лише з теки `EPS_2026/demo_pairs` на Google Drive (тільки читання — ноутбук туди нічого не записує): пара `<ім'я>_pre.png` + `<ім'я>_post.png` або `<ім'я>_before.png` + `<ім'я>_after.png` (чи просто `before.png` + `after.png`), розмітка `<ім'я>_label.png` (необов'язково; тоді інтерфейс показує F1 кадру). Підтеки теж читаються, крім службових, що починаються з `_` (напр. `_old`). Щоб змінити набір прикладів — додайте/видаліть файли в `demo_pairs` і перезапустіть комірки 5–7.

**Якщо Gradio недоступний:** `res = run_detection('pre.png', 'post.png', gsd=0.3); show_result(res)` — той самий результат у matplotlib.